# Phase 2 local real-hardware runner (standalone)

This notebook is self-contained: it does **not** import modules from this repository.

You only need Python packages installed in your local environment (`qiskit`, `qiskit-ibm-runtime`, `qiskit-algorithms`, `numpy`, `matplotlib`).

It writes each run under:

- `outputs/<timestamp>_phase2_local-real-hw/<architecture>/seed_<seed>_qubits_<q>/optimizer_history.npz`
- `outputs/<timestamp>_phase2_local-real-hw/<architecture>/seed_<seed>_qubits_<q>/run_status.json`
- `outputs/<timestamp>_phase2_local-real-hw/<architecture>/seed_<seed>_qubits_<q>/optimizer_compare.png`
- `outputs/<timestamp>_phase2_local-real-hw/<architecture>/seed_<seed>_qubits_<q>/optimizer_compare_budget.png`

This output format is compatible with your current analysis flow after copying the folder back to the cluster.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import json
import os
import traceback

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_algorithms.optimizers import COBYLA, QNSPSA
from qiskit_ibm_runtime import Estimator as RuntimeEstimator
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.utils.validations import validate_isa_circuits

PROJECT_ROOT = Path.cwd().resolve()
print("PROJECT_ROOT:", PROJECT_ROOT)


class ConvergenceReached(Exception):
    pass


class BudgetExceeded(Exception):
    pass


@dataclass
class ArchitectureSpec:
    circuit: QuantumCircuit
    readout_qubit: int


def parse_depth_split(value: str) -> tuple[int, int]:
    parts = [chunk.strip() for chunk in value.split(",")]
    if len(parts) != 2:
        raise ValueError("resqnet_depth_split must have format 'D1,D2'")
    d1, d2 = int(parts[0]), int(parts[1])
    if d1 < 1 or d2 < 1:
        raise ValueError("resqnet depths must be positive")
    return d1, d2


def load_env_file(path: str = ".env") -> None:
    p = Path(path)
    if not p.is_file():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        raw = line.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_env_first(*names: str) -> str | None:
    for name in names:
        value = os.environ.get(name)
        if value:
            return value
    return None


def resolve_budget_evals(
    *,
    num_params: int,
    budget_evals: int | None,
    budget_k: float | None,
    min_budget_evals: int,
    max_budget_evals: int | None,
) -> tuple[int, str]:
    if budget_evals is not None:
        resolved = int(budget_evals)
        mode = "fixed"
    else:
        if budget_k is None:
            raise ValueError("budget_k is required when budget_evals is None")
        resolved = int(round(float(budget_k) * int(num_params)))
        mode = "dynamic"

    resolved = max(int(min_budget_evals), resolved)
    if max_budget_evals is not None:
        resolved = min(int(max_budget_evals), resolved)
    return resolved, mode


def init_trackers(*names: str):
    history = {name: {"evals": [], "cost": []} for name in names}
    counts = {name: 0 for name in names}
    return history, counts


def make_objective(
    name: str,
    *,
    counts: dict,
    history: dict,
    estimator,
    ansatz,
    observable,
    budget_evals: int | None,
    result_timeout_s: float | None,
):
    def objective(parameters):
        next_count = counts[name] + 1
        if budget_evals is not None and next_count > budget_evals:
            raise BudgetExceeded()

        counts[name] = next_count
        pub = (ansatz, observable, [parameters])
        job = estimator.run([pub])
        if result_timeout_s is None:
            result = job.result()[0]
        else:
            try:
                result = job.result(timeout=result_timeout_s)[0]
            except TypeError:
                result = job.result()[0]

        cost = float(result.data.evs[0])
        history[name]["evals"].append(counts[name])
        history[name]["cost"].append(cost)
        return cost

    return objective


def with_early_stopping(objective, *, history: dict, key: str, window: int, tolerance: float):
    def wrapped(parameters):
        cost = objective(parameters)
        if len(history[key]["cost"]) >= window:
            last_window = history[key]["cost"][-window:]
            if max(last_window) - min(last_window) < tolerance:
                raise ConvergenceReached()
        return cost

    return wrapped


def make_fidelity(*, counts: dict, key: str, circuit, budget_evals: int | None):
    def fidelity(params1, params2):
        next_count = counts[key] + 2
        if budget_evals is not None and next_count > budget_evals:
            raise BudgetExceeded()
        counts[key] = next_count
        sv1 = Statevector(circuit.assign_parameters(params1))
        sv2 = Statevector(circuit.assign_parameters(params2))
        return np.abs(sv1.inner(sv2)) ** 2

    return fidelity


def run_optimizer(name: str, optimizer, objective, x0, *, counts: dict):
    print(f"Running {name}...")
    try:
        optimizer.minimize(fun=objective, x0=x0)
    except BudgetExceeded:
        print(f"-> {name} stopped at budget after eval {counts[name]}.")
    except ConvergenceReached:
        print(f"-> {name} stopped early at eval {counts[name]}.")


def save_optimizer_time_series(history: dict, outdir: str, *, filename: str = "optimizer_compare.png") -> None:
    os.makedirs(outdir, exist_ok=True)
    stem, ext = os.path.splitext(filename)
    if not ext:
        ext = ".png"

    def _plot(mode: str, xlabel: str, target_name: str) -> None:
        plt.figure(figsize=(8, 5))
        if "cobyla" in history:
            x = list(range(1, len(history["cobyla"]["cost"]) + 1)) if mode == "iter" else history["cobyla"]["evals"]
            plt.plot(x, history["cobyla"]["cost"], label="COBYLA", color="blue")
        if "qnspsa" in history:
            x = list(range(1, len(history["qnspsa"]["cost"]) + 1)) if mode == "iter" else history["qnspsa"]["evals"]
            plt.plot(x, history["qnspsa"]["cost"], label="QNSPSA", color="green")

        plt.axhline(y=-1.0, color="r", linestyle="--", label="theoretical min -1")
        plt.xlabel(xlabel)
        plt.ylabel("Cost")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(Path(outdir) / target_name)
        plt.close()

    _plot("iter", "Optimizer iteration", f"{stem}{ext}")
    _plot("eval", "Eval count", f"{stem}_budget{ext}")


def _build_baseline_hea(n_qubits: int, depth: int = 4) -> ArchitectureSpec:
    from qiskit.circuit import ParameterVector

    params = ParameterVector("th", length=2 * depth * n_qubits)
    qc = QuantumCircuit(n_qubits)
    idx = 0
    for _layer in range(depth):
        for q in range(n_qubits):
            qc.ry(params[idx], q)
            idx += 1
        for q in range(n_qubits):
            qc.rz(params[idx], q)
            idx += 1
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        if n_qubits > 2:
            qc.cx(n_qubits - 1, 0)
    return ArchitectureSpec(circuit=qc, readout_qubit=0)


def _build_qcnn(n_qubits: int, depth: int = 3) -> ArchitectureSpec:
    from qiskit.circuit import ParameterVector

    params = ParameterVector("qcnn", length=4 * depth * n_qubits)
    qc = QuantumCircuit(n_qubits)
    idx = 0
    for _ in range(depth):
        for q in range(0, n_qubits - 1, 2):
            qc.ry(params[idx], q)
            idx += 1
            qc.ry(params[idx], q + 1)
            idx += 1
            qc.cx(q, q + 1)
            qc.rz(params[idx], q + 1)
            idx += 1
            qc.cx(q, q + 1)
        for q in range(1, n_qubits - 1, 2):
            qc.ry(params[idx], q)
            idx += 1
            qc.ry(params[idx], q + 1)
            idx += 1
            qc.cx(q, q + 1)
            qc.rz(params[idx], q + 1)
            idx += 1
            qc.cx(q, q + 1)
    return ArchitectureSpec(circuit=qc, readout_qubit=0)


def _build_resqnet(n_qubits: int, depth_split: tuple[int, int], residual_mode: str) -> ArchitectureSpec:
    if residual_mode != "structural":
        raise ValueError("Only residual_mode='structural' is supported")

    from qiskit.circuit import ParameterVector

    d1, d2 = depth_split
    params = ParameterVector("resq", length=2 * (d1 + d2) * n_qubits)
    qc = QuantumCircuit(n_qubits)
    idx = 0

    for _ in range(d1):
        for q in range(n_qubits):
            qc.rx(params[idx], q)
            idx += 1
            qc.ry(params[idx], q)
            idx += 1
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)

    # Structural residual link between the two blocks.
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)

    for _ in range(d2):
        for q in range(n_qubits):
            qc.rx(params[idx], q)
            idx += 1
            qc.ry(params[idx], q)
            idx += 1
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)

    return ArchitectureSpec(circuit=qc, readout_qubit=0)


def build_architecture(
    *,
    architecture: str,
    n_qubits: int,
    resqnet_depth_split: tuple[int, int],
    resqnet_residual_mode: str,
) -> ArchitectureSpec:
    if architecture == "baseline_hea":
        return _build_baseline_hea(n_qubits)
    if architecture == "qcnn":
        return _build_qcnn(n_qubits)
    if architecture == "resqnet":
        return _build_resqnet(n_qubits, depth_split=resqnet_depth_split, residual_mode=resqnet_residual_mode)
    raise ValueError(f"Unknown architecture: {architecture}")


def build_runtime_service(channel: str, token: str, instance: str | None) -> QiskitRuntimeService:
    channels_to_try = [channel] if channel in ["ibm_cloud", "ibm_quantum_platform", "ibm_quantum"] else ["ibm_quantum_platform", "ibm_cloud", "ibm_quantum"]
    last_error = None
    for ch in channels_to_try:
        try:
            kwargs = {"channel": ch, "token": token}
            if instance:
                kwargs["instance"] = instance
            return QiskitRuntimeService(**kwargs)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Failed to connect to IBM Quantum. Last error: {last_error}")


def optimizer_compare(
    *,
    n_qubits: int,
    outdir: str,
    seed: int | None,
    budget_evals: int | None,
    budget_k: float | None,
    window: int,
    tol: float,
    shots: int,
    backend_name: str | None,
    channel: str,
    optimization_level: int | None,
    resilience_level: int | None,
    runtime_result_timeout: float,
    architecture: str,
    resqnet_depth_split: tuple[int, int],
    resqnet_residual_mode: str,
) -> None:
    min_budget_evals = 1
    max_budget_evals = 4000

    if seed is not None:
        np.random.seed(seed)

    arch = build_architecture(
        architecture=architecture,
        n_qubits=n_qubits,
        resqnet_depth_split=resqnet_depth_split,
        resqnet_residual_mode=resqnet_residual_mode,
    )
    ansatz_logical = arch.circuit
    observable_logical = SparsePauliOp.from_list(
        [("I" * arch.readout_qubit + "Z" + "I" * (n_qubits - arch.readout_qubit - 1), 1.0)]
    )

    load_env_file(".env")
    token = get_env_first("QISKIT_IBM_TOKEN", "API_KEY")
    instance = get_env_first("QISKIT_IBM_INSTANCE", "CRN_KEY")
    if not token:
        raise RuntimeError("Missing IBM token. Set QISKIT_IBM_TOKEN or API_KEY in .env/environment.")

    print("Runtime preflight: creating IBM Runtime service...")
    service = build_runtime_service(channel=channel, token=token, instance=instance)
    print("Runtime preflight: selecting backend...")
    backend = service.backend(backend_name) if backend_name else service.least_busy(simulator=False, min_num_qubits=n_qubits)
    print(f"Runtime preflight: selected backend={backend.name}")

    opt_level = 1 if optimization_level is None else optimization_level
    pm = generate_preset_pass_manager(backend=backend, optimization_level=opt_level)
    ansatz = pm.run(ansatz_logical)
    observable = observable_logical.apply_layout(ansatz.layout)
    validate_isa_circuits([ansatz], backend.target)

    logical_num_params = ansatz_logical.num_parameters
    isa_num_params = ansatz.num_parameters
    if logical_num_params != isa_num_params:
        raise RuntimeError(
            f"Parameter mismatch between logical and ISA ansatz (logical={logical_num_params}, isa={isa_num_params})."
        )

    resolved_budget_evals, budget_mode = resolve_budget_evals(
        num_params=logical_num_params,
        budget_evals=budget_evals,
        budget_k=budget_k,
        min_budget_evals=min_budget_evals,
        max_budget_evals=max_budget_evals,
    )
    print(
        f"Architecture={architecture} | Budget mode: {budget_mode} | num_params={logical_num_params} | "
        f"k={budget_k} | budget_evals={resolved_budget_evals}"
    )

    initial_parameters = np.random.normal(loc=0.0, scale=0.1, size=logical_num_params)

    options = EstimatorOptions()
    options.default_shots = shots
    if resilience_level is not None:
        options.resilience_level = resilience_level

    history, counts = init_trackers("cobyla", "qnspsa")
    run_status = {
        "status": "failed",
        "backend": backend.name,
        "architecture": architecture,
        "runtime_result_timeout_s": runtime_result_timeout,
        "error": None,
    }

    try:
        with Session(backend=backend) as session:
            estimator = RuntimeEstimator(mode=session, options=options)

            objective_cobyla = with_early_stopping(
                make_objective(
                    "cobyla",
                    counts=counts,
                    history=history,
                    estimator=estimator,
                    ansatz=ansatz,
                    observable=observable,
                    budget_evals=resolved_budget_evals,
                    result_timeout_s=runtime_result_timeout,
                ),
                history=history,
                key="cobyla",
                window=window,
                tolerance=tol,
            )
            objective_qnspsa = with_early_stopping(
                make_objective(
                    "qnspsa",
                    counts=counts,
                    history=history,
                    estimator=estimator,
                    ansatz=ansatz,
                    observable=observable,
                    budget_evals=resolved_budget_evals,
                    result_timeout_s=runtime_result_timeout,
                ),
                history=history,
                key="qnspsa",
                window=window,
                tolerance=tol,
            )

            fidelity = make_fidelity(
                counts=counts,
                key="qnspsa",
                circuit=ansatz_logical,
                budget_evals=resolved_budget_evals,
            )

            run_optimizer(
                "cobyla",
                COBYLA(maxiter=resolved_budget_evals),
                objective_cobyla,
                initial_parameters,
                counts=counts,
            )
            run_optimizer(
                "qnspsa",
                QNSPSA(fidelity=fidelity, maxiter=resolved_budget_evals),
                objective_qnspsa,
                initial_parameters,
                counts=counts,
            )

        run_status["status"] = "completed"
    except Exception as exc:
        run_status["status"] = "failed_runtime"
        run_status["error"] = repr(exc)
        print(f"Runtime execution failed: {exc!r}")
        raise
    finally:
        os.makedirs(outdir, exist_ok=True)
        np.savez(os.path.join(outdir, "optimizer_history.npz"), history=history)
        if history["cobyla"]["cost"] or history["qnspsa"]["cost"]:
            save_optimizer_time_series(history, outdir)
        with open(os.path.join(outdir, "run_status.json"), "w", encoding="utf-8") as handle:
            json.dump(run_status, handle, indent=2, sort_keys=True)
        print("Saved optimizer results to", outdir)


In [ ]:
# ---------- Local experiment config ----------
RUN_TAG = datetime.now().strftime("%Y%m%d-%H%M%S") + "_phase2_local-real-hw"
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_TAG

ARCHITECTURES = ["qcnn"]  # e.g. ["baseline_hea", "qcnn", "resqnet"]
SEEDS = [10]
QUBITS = [4]

COMMON = {
    "budget_evals": None,
    "budget_k": 10.0,
    "window": 30,
    "tol": 1e-3,
    "shots": 256,
    "backend_name": "ibm_basquecountry",  # set None to use least_busy
    "channel": "ibm_quantum_platform",
    "optimization_level": 1,
    "resilience_level": None,
    "runtime_result_timeout": 120.0,
    "resqnet_depth_split": parse_depth_split("5,1"),
    "resqnet_residual_mode": "structural",
}

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("RUN_ROOT:", RUN_ROOT)


In [ ]:
# ---------- Execute runs ----------
records: list[dict] = []

for architecture in ARCHITECTURES:
    for seed in SEEDS:
        for n_qubits in QUBITS:
            outdir = RUN_ROOT / architecture / f"seed_{seed}_qubits_{n_qubits}"
            print(f"\n=== RUN architecture={architecture} seed={seed} qubits={n_qubits} ===")
            try:
                optimizer_compare(
                    n_qubits=n_qubits,
                    outdir=str(outdir),
                    seed=seed,
                    architecture=architecture,
                    **COMMON,
                )
                status = "completed"
                error = None
            except Exception as exc:
                status = "failed"
                error = repr(exc)
                print("RUN FAILED:", error)
                traceback.print_exc()

            records.append(
                {
                    "architecture": architecture,
                    "seed": seed,
                    "n_qubits": n_qubits,
                    "outdir": str(outdir),
                    "status": status,
                    "error": error,
                }
            )

print("\nDone. Total runs:", len(records))


In [ ]:
# ---------- Save local manifest ----------
manifest_path = RUN_ROOT / "local_notebook_manifest.json"
manifest = {
    "run_tag": RUN_TAG,
    "run_root": str(RUN_ROOT),
    "config": {
        "architectures": ARCHITECTURES,
        "seeds": SEEDS,
        "qubits": QUBITS,
        "common": {
            **COMMON,
            "resqnet_depth_split": list(COMMON["resqnet_depth_split"]),
        },
    },
    "records": records,
}

manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest saved to:", manifest_path)
print("\nNext step:")
print("1) Copy this RUN_ROOT folder to your cluster workspace under outputs/")
print("2) Run your usual analyze script there")
